# Практическая работа. Емкость модели, переобучение и регуляризация
## «Выучить наизусть – не значит понять»

**Цель:** построить своей кросс-валидацией кривые валидации и обучения для kNN и дерева решений, разложить ошибку kNN-регрессии на смещение и разброс, освоить бутстрэп, добавить к логистической регрессии из Б1.3 штраф L2 и подобрать его силу, а в конце остановить обучение до того, как модель начнет зубрить.

> **Правило работы.** Внутри своих функций – только списки, словари, циклы, `math` и `random`. Библиотека `sklearn` нужна только для загрузки Wine и сверки в задании Р6, `matplotlib` – только в готовых ячейках с графиками.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. Напечатанные значения должны совпасть с тем, что стоит после слова «ожидается». Готовые ячейки просто выполните. Сначала прочитайте теорию.

**План работы**

| Задание | Что пишем | Главный результат |
|---|---|---|
| Р1 | `validation_curve` | кривые валидации для $k$ в kNN и глубины дерева |
| Р2 | `learning_curve` | кривые обучения: дерево глубины 2 и глубины 12 |
| Р3 | `predictions_at`, `bias_variance` | смещение и разброс kNN-регрессии на генераторе |
| Р4 | `bootstrap_sample` | разброс прогнозов по одной выборке |
| Р5 | `gradient_l2`, `fit_l2` | логистическая регрессия со штрафом L2 |
| Р6 | `poly_features`, `cv_logreg` | лишняя емкость и подбор силы штрафа $\lambda$ |
| Р7 | `fit_early_stopping` | ранняя остановка обучения |

## Теория: емкость, смещение и разброс, регуляризация

**1. Обобщение и две ошибки.** Модель ценна, если хорошо работает на новых объектах, – это **обобщающая способность** (Б1.1). У модели две ошибки: **ошибка обучения** (на обучающей выборке) и **ошибка обобщения** (на новых данных; ее оценивают кросс-валидацией или тестом). Хорошая модель одновременно делает ошибку обучения малой и держит маленьким **разрыв** между двумя ошибками. Отсюда две беды:

- **недообучение:** модель плохо работает даже на обучающей выборке;
- **переобучение:** на обучающей выборке все хорошо, а разрыв с новыми данными велик.

**2. Емкость модели** – насколько разнообразные правила модель способна выучить. Емкость kNN тем больше, чем меньше $k$ (при $k = 1$ граница повторяет каждую точку); емкость дерева растет с глубиной; емкость логистической регрессии растет, если добавить степени признаков ($x_1^2$, $x_1 x_2$, $x_2^3$, …): тогда граница из прямой превращается в кривую. Малая емкость ведет к недообучению, избыточная – к переобучению. Лучше всего работает емкость, которая соответствует сложности задачи и объему данных.

**3. Кривая валидации** – точность на обучающей части и на валидации (кросс-валидация) в зависимости от гиперпараметра, управляющего емкостью. Слева (малая емкость) обе кривые низкие и близки – недообучение. Справа (большая емкость) обучающая кривая уходит к 100 %, а валидационная перестает расти или падает – переобучение. Лучший гиперпараметр – вершина валидационной кривой.

**4. Кривая обучения** – те же две точности в зависимости от **размера** обучающей выборки при фиксированной модели. Если кривые сошлись на низком уровне, больше данных не поможет – нужна модель сложнее. Если между ними большой разрыв, помогут новые данные или упрощение модели.

**5. Смещение и разброс.** Представим, что мы много раз собрали новую обучающую выборку из того же источника и каждый раз обучили модель. Прогнозы для одной и той же точки $x$ будут разными.

- **Смещение** – насколько **средний** прогноз отличается от истинного значения $f(x)$.
- **Разброс** (дисперсия) – насколько прогнозы разбегаются вокруг своего среднего.

Для среднеквадратичной ошибки на новых данных верно разложение (Гудфеллоу, формула 5.54, плюс шум данных):

$$\mathbb{E}\big[(y - \hat y)^2\big] = \underbrace{\big(\bar{\hat y} - f(x)\big)^2}_{\text{смещение}^2} + \underbrace{\mathbb{E}\big[(\hat y - \bar{\hat y})^2\big]}_{\text{разброс}} + \underbrace{\sigma^2_{\text{шум}}}_{\text{шум}},$$

где $\bar{\hat y}$ – средний прогноз по многим обучающим выборкам. С ростом емкости смещение падает, а разброс растет, поэтому ошибка имеет форму буквы U. Аналогия с мишенью: большое смещение – все выстрелы кучно, но мимо центра; большой разброс – в среднем в центр, но выстрелы разбросаны.

**6. Бутстрэп.** В жизни генератора данных нет, есть одна выборка из $n$ объектов. Новые выборки можно имитировать: вытянуть $n$ объектов **с возвращением** (один объект может попасть несколько раз, а другой ни разу). В бутстрэп-выборку попадает в среднем $1 - (1 - 1/n)^n \approx 1 - 1/e \approx 63\,\%$ различных объектов. Обучая модель на многих бутстрэп-выборках, оцениваем разброс ее прогнозов. На этой же идее построен бэггинг (задание повышенной сложности).

**7. L2-регуляризация** (снижение весов). К функции потерь логистической регрессии добавляем штраф за большие веса:

$$J(w, b) = L(w, b) + \frac{\lambda}{2}\sum_{j=1}^{n} w_j^2,$$

где $L$ – log-loss из Б1.3, $\lambda \ge 0$ – **сила регуляризации** (гиперпараметр). Сдвиг $b$ не штрафуют. Градиент получает одно слагаемое:

$$\frac{\partial J}{\partial w_j} = \frac{\partial L}{\partial w_j} + \lambda w_j, \qquad w_j \leftarrow w_j - \eta\Big(\frac{\partial L}{\partial w_j} + \lambda w_j\Big) = (1 - \eta\lambda)\,w_j - \eta\,\frac{\partial L}{\partial w_j}.$$

То есть перед обычным шагом каждый вес умножается на число $1 - \eta\lambda$, меньшее 1: веса «усыхают», если данные не требуют обратного. При $\lambda = 0$ штрафа нет; при большом $\lambda$ веса почти нулевые, и модель недообучается. Малые веса дают более плавную границу и менее самоуверенные вероятности. Силу $\lambda$ выбирают кросс-валидацией, как любой гиперпараметр. В sklearn штраф записан через $C$: `LogisticRegression(C=1 / (λ · n))`, где $n$ – число обучающих объектов.

**8. Ранняя остановка.** При обучении градиентным спуском ошибка на обучающей выборке падает все время, а на валидационной сначала падает, потом растет: модель начинает подгоняться под шум. Ранняя остановка запоминает веса в эпоху с наименьшей ошибкой на валидации и прекращает обучение, если улучшения нет уже `patience` эпох подряд. Число эпох становится еще одним гиперпараметром, управляющим емкостью.

**Где почитать.** Я. Гудфеллоу, И. Бенджио, А. Курвилль «Глубокое обучение»: 5.2 «Емкость, переобучение и недообучение», 5.2.2 «Регуляризация», 5.4.4 «Поиск компромисса между смещением и дисперсией», 7.1.1 «Регуляризация параметров по норме L2», 7.8 «Ранняя остановка», 7.11 «Баггинг и другие ансамблевые методы».

In [ ]:
# Готовая ячейка: библиотеки, данные и функции из Б2.1
import math
import random
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine     # только для загрузки набора Wine

RANDOM_STATE = 42
BLUE, NAVY, GREY, DARK_BLUE = '#4C9BFF', '#0F1B2D', '#969CA3', '#1F63C9'

def train_test_split(X, y, test_size=0.3, seed=RANDOM_STATE):
    idx = list(range(len(X)))
    random.Random(seed).shuffle(idx)
    n_test = int(len(X) * test_size)
    test, train = idx[:n_test], idx[n_test:]
    return ([X[i] for i in train], [X[i] for i in test],
            [y[i] for i in train], [y[i] for i in test])

def make_road_objects(n_per_class=100, seed=RANDOM_STATE):
    """Объекты на дороге (Б1.2): 0 – пешеход, 1 – велосипедист, 2 – автомобиль."""
    rnd = random.Random(seed)
    params = {0: ((5, 2.5), (0.7, 0.4), (1.7, 0.15)),
              1: ((14, 6), (1.7, 0.45), (1.75, 0.15)),
              2: ((18, 12), (3.6, 1.1), (1.6, 0.2))}
    X, y = [], []
    for c, (pv, pl, ph) in params.items():
        for _ in range(n_per_class):
            v = max(0.5, rnd.gauss(*pv)); L = max(0.2, rnd.gauss(*pl)); H = max(0.8, rnd.gauss(*ph))
            X.append([round(v, 1), round(L, 2), round(H, 2), round(rnd.uniform(5, 80), 1)])
            y.append(c)
    idx = list(range(len(X)))
    rnd.shuffle(idx)
    return [X[i] for i in idx], [y[i] for i in idx]

def make_braking(n=80, seed=RANDOM_STATE):
    """Скорость (км/ч) -> тормозной путь (м) с шумом (Б1.2)."""
    rnd = random.Random(seed)
    X, y = [], []
    for _ in range(n):
        v = rnd.uniform(10, 120)
        X.append([round(v, 1)])
        y.append(round(true_braking(v) + rnd.gauss(0, 3 + 0.05 * v), 1))
    return X, y

def true_braking(v):
    """Истинный тормозной путь без шума: реакция 1 с плюс торможение."""
    return v / 3.6 + v ** 2 / (254 * 0.7)

# ---- функции из Б2.1
def mean_std(values):
    m = sum(values) / len(values)
    return m, math.sqrt(sum((v - m) ** 2 for v in values) / len(values))

def split_by_fold(X, y, folds, i):
    train_idx = [j for f in range(len(folds)) if f != i for j in folds[f]]
    return ([X[j] for j in train_idx], [y[j] for j in train_idx],
            [X[j] for j in folds[i]], [y[j] for j in folds[i]])

def stratified_k_fold_indices(y, n_folds, seed=RANDOM_STATE):
    rnd = random.Random(seed)
    by_class = {}
    for i, label in enumerate(y):
        by_class.setdefault(label, []).append(i)
    folds, pos = [[] for _ in range(n_folds)], 0
    for label in sorted(by_class):
        idx = by_class[label]
        rnd.shuffle(idx)
        for i in idx:
            folds[pos % n_folds].append(i)
            pos += 1
    return folds

def accuracy(y_true, y_pred):
    return sum(1 for a, b in zip(y_true, y_pred) if a == b) / len(y_true)

def minmax_fit(X):
    return ([min(r[j] for r in X) for j in range(len(X[0]))], [max(r[j] for r in X) for j in range(len(X[0]))])

def minmax_transform(X, mins, maxs):
    return [[(v - lo) / (hi - lo) for v, lo, hi in zip(row, mins, maxs)] for row in X]

# Данные беспилотника, как в Б2.1: 210 объектов, три признака после нормализации min-max
X_road, y_road = make_road_objects()
X_train, X_test, y_train, y_test = train_test_split(X_road, y_road)
X_train3, X_test3 = [r[:3] for r in X_train], [r[:3] for r in X_test]
mins, maxs = minmax_fit(X_train3)
Xr_train, Xr_test = minmax_transform(X_train3, mins, maxs), minmax_transform(X_test3, mins, maxs)
folds_r = stratified_k_fold_indices(y_train, 5)

# Тормозной путь, как в Б1.2: 56 замеров в обучающей выборке
X_b, y_b = make_braking()
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_b, y_b)
print('беспилотник:', len(Xr_train), ' тормозной путь:', len(Xb_train), '  ожидается 210 и 56')

In [ ]:
# Готовая ячейка: модели из прошлых работ
# ---- kNN (Б1.2)
def euclidean(a, b):
    return sum((ai - bi) ** 2 for ai, bi in zip(a, b)) ** 0.5

def get_neighbors(X_train, y_train, x, k):
    pairs = sorted(((euclidean(xi, x), yi) for xi, yi in zip(X_train, y_train)), key=lambda p: p[0])
    return [yi for d, yi in pairs[:k]]

def majority_vote(labels):
    counts = {}
    for lab in labels:
        counts[lab] = counts.get(lab, 0) + 1
    best = max(counts.values())
    for lab in labels:
        if counts[lab] == best:
            return lab

def knn_predict(X_train, y_train, X_test, k):
    return [majority_vote(get_neighbors(X_train, y_train, x, k)) for x in X_test]

def knn_regress(X_train, y_train, X_test, k):
    """kNN-регрессия: среднее ответов k ближайших соседей."""
    result = []
    for x in X_test:
        values = get_neighbors(X_train, y_train, x, k)
        result.append(sum(values) / len(values))
    return result

# ---- дерево решений (Б1.4)
def count_classes(y):
    counts = {}
    for label in y:
        counts[label] = counts.get(label, 0) + 1
    return counts

def gini(y):
    n = len(y)
    return 0.0 if n == 0 else 1 - sum((c / n) ** 2 for c in count_classes(y).values())

def split(X, y, feature, threshold):
    XL, yL, XR, yR = [], [], [], []
    for x, label in zip(X, y):
        if x[feature] <= threshold:
            XL.append(x); yL.append(label)
        else:
            XR.append(x); yR.append(label)
    return XL, yL, XR, yR

def best_split(X, y):
    best_feature, best_threshold, best_gain = None, None, 0.0
    for feature in range(len(X[0])):
        values = sorted(set(x[feature] for x in X))
        for t in [(values[i] + values[i + 1]) / 2 for i in range(len(values) - 1)]:
            _, yl, _, yr = split(X, y, feature, t)
            n = len(y)
            gain = gini(y) - (len(yl) / n * gini(yl) + len(yr) / n * gini(yr))
            if gain > best_gain:
                best_feature, best_threshold, best_gain = feature, t, gain
    return best_feature, best_threshold, best_gain

def build_tree(X, y, depth=0, max_depth=3):
    counts = count_classes(y)
    leaf = {'leaf': True, 'class': max(counts, key=counts.get)}
    if depth >= max_depth or len(y) < 2 or len(counts) == 1:
        return leaf
    feature, threshold, gain = best_split(X, y)
    if feature is None:
        return leaf
    XL, yL, XR, yR = split(X, y, feature, threshold)
    return {'leaf': False, 'feature': feature, 'threshold': threshold,
            'left': build_tree(XL, yL, depth + 1, max_depth), 'right': build_tree(XR, yR, depth + 1, max_depth)}

def predict_tree(tree, X):
    result = []
    for x in X:
        node = tree
        while not node['leaf']:
            node = node['left'] if x[node['feature']] <= node['threshold'] else node['right']
        result.append(node['class'])
    return result

# ---- обертки по договору model(X_train, y_train, X_val, params) из Б2.1
def knn_model(X_train, y_train, X_val, params):
    return knn_predict(X_train, y_train, X_val, params['k'])

def tree_model(X_train, y_train, X_val, params):
    return predict_tree(build_tree(X_train, y_train, max_depth=params['max_depth']), X_val)

print('модели готовы')

**Задание Р1. Кривая валидации.**

`validation_curve(model, X, y, folds, name, values)` – для каждого значения `v` гиперпараметра с названием `name` (словарь `params = {name: v}`) пройти по всем раундам кросс-валидации и посчитать две точности:

- на **обучающей части** раунда: модель обучена на `X_tr, y_tr` и предсказывает те же `X_tr`;
- на **валидационном блоке**: модель обучена на `X_tr, y_tr` и предсказывает `X_val`.

Средние по раундам сложите в списки `train_scores` и `val_scores`. Функция возвращает пару этих списков.

Затем постройте кривые для числа соседей $k$ в kNN и для глубины дерева на данных беспилотника.

In [ ]:
def validation_curve(model, X, y, folds, name, values):
    train_scores, val_scores = [], []
    for v in values:
        params = {name: v}
        tr, va = [], []
        for i in range(len(folds)):
            X_tr, y_tr, X_val, y_val = split_by_fold(X, y, folds, i)
            ...                                 # <-- точность на обучающей части раунда
            ...                                 # <-- точность на валидационном блоке
        ...                                     # <-- среднее по раундам
        ...                                     # <-- среднее по раундам
    return train_scores, val_scores


ks = [1, 3, 5, 9, 13, 21, 31, 51, 71, 99]
knn_train, knn_val = validation_curve(knn_model, Xr_train, y_train, folds_r, 'k', ks)
for k, a, b in zip(ks, knn_train, knn_val):
    if k in (1, 13, 99):
        print(f'k = {k:2d}   обучение {a:.3f}   валидация {b:.3f}')
print('  ожидается: k = 1: 1.000 и 0.829;  k = 13: 0.889 и 0.890;  k = 99: 0.769 и 0.767')

depths = list(range(1, 13))
tree_train, tree_val = validation_curve(tree_model, Xr_train, y_train, folds_r, 'max_depth', depths)
for d, a, b in zip(depths, tree_train, tree_val):
    if d in (1, 6, 12):
        print(f'глубина {d:2d}   обучение {a:.3f}   валидация {b:.3f}')
print('  ожидается: 1: 0.601 и 0.586;  6: 0.951 и 0.876;  12: 1.000 и 0.867')

In [ ]:
# Готовая ячейка: две кривые валидации
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ks, knn_train, 'o-', color=GREY, label='обучающая часть')
ax[0].plot(ks, knn_val, 'o-', color=BLUE, label='валидация (CV)')
ax[0].set_xlabel('k – число соседей  (емкость падает →)'); ax[0].set_title('kNN')
ax[1].plot(depths, tree_train, 'o-', color=GREY, label='обучающая часть')
ax[1].plot(depths, tree_val, 'o-', color=BLUE, label='валидация (CV)')
ax[1].set_xlabel('max_depth  (емкость растет →)'); ax[1].set_title('Дерево решений')
for a in ax:
    a.set_ylabel('точность'); a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.show()

**Вопрос Р1.** а) Почему при $k = 1$ точность на обучающей части ровно 1,000? б) Где на каждой кривой зона недообучения, а где переобучения? Как вы это определили? в) У дерева с глубины 6 точность на обучающей части растет до 1,000, а на валидации стоит на месте. Что дерево выучивает на глубинах 7–12?

*Ваш ответ:*

**Задание Р2. Кривая обучения.**

`learning_curve(model, X, y, folds, params, sizes)` – для каждого размера `m` из списка `sizes` в каждом раунде кросс-валидации взять из обучающей части **случайные** `m` объектов, обучить на них модель и посчитать точность на этих `m` объектах и на валидационном блоке. Случайный набор номеров дает `random.Random(RANDOM_STATE + i).sample(range(len(X_tr)), m)`: разные раунды получают разные наборы, а при повторном запуске результат тот же. Брать первые `m` объектов нельзя: в стратифицированных блоках номера идут по классам, и первые объекты могли бы оказаться одного класса.

Сравните два дерева: мелкое (глубина 2) и глубокое (глубина 12).

In [ ]:
def learning_curve(model, X, y, folds, params, sizes):
    train_scores, val_scores = [], []
    for m in sizes:
        tr, va = [], []
        for i in range(len(folds)):
            X_tr, y_tr, X_val, y_val = split_by_fold(X, y, folds, i)
            sub = ...                           # <-- m случайных номеров
            X_m = ...                           # <-- объекты подвыборки
            y_m = ...                           # <-- ответы подвыборки
            ...                                 # <-- точность на подвыборке
            ...                                 # <-- точность на валидации
        train_scores.append(mean_std(tr)[0])
        val_scores.append(mean_std(va)[0])
    return train_scores, val_scores


sizes = [20, 40, 60, 80, 100, 120, 140, 168]
lc = {}
for d in (2, 12):
    lc[d] = learning_curve(tree_model, Xr_train, y_train, folds_r, {'max_depth': d}, sizes)
    tr, va = lc[d]
    print(f'глубина {d:2d}:  m = 20: {tr[0]:.3f} и {va[0]:.3f};   m = 168: {tr[-1]:.3f} и {va[-1]:.3f}')
print('  ожидается: глубина 2: 0.940 и 0.719, затем 0.844 и 0.838;  глубина 12: 1.000 и 0.695, затем 1.000 и 0.867')

In [ ]:
# Готовая ячейка: кривые обучения двух деревьев
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for a, d, title in [(ax[0], 2, 'глубина 2: большое смещение'), (ax[1], 12, 'глубина 12: большой разброс')]:
    tr, va = lc[d]
    a.plot(sizes, tr, 'o-', color=GREY, label='обучающая подвыборка')
    a.plot(sizes, va, 'o-', color=BLUE, label='валидация (CV)')
    a.fill_between(sizes, va, tr, color=BLUE, alpha=0.08)
    a.set_xlabel('размер обучающей выборки m'); a.set_title(title); a.grid(alpha=0.3); a.legend()
ax[0].set_ylabel('точность'); plt.tight_layout(); plt.show()

**Вопрос Р2.** а) Почему у мелкого дерева точность на обучающей подвыборке **падает** с ростом $m$? б) Для какого дерева имеет смысл собирать больше данных, а для какого – нет? Обоснуйте по кривым. в) Что нужно сделать с мелким деревом, чтобы поднять качество?

*Ваш ответ:*

**Задание Р3. Смещение и разброс kNN-регрессии.**

Вернемся к тормозному пути из Б1.2. У нас есть редкая роскошь: известна **истинная** зависимость `true_braking(v)` и генератор `make_braking(n, seed)`, который может выдать сколько угодно новых выборок из того же источника. Поэтому смещение и разброс можно посчитать по определению из п. 5 теории.

- `predictions_at(v0, k, n_sets=100, n=56)` – для `seed` от 1000 до 1099 получить новую выборку из 56 замеров, обучить kNN-регрессию (для kNN это просто запомнить выборку) и сделать прогноз для одной скорости `v0`. Возвращает список из `n_sets` прогнозов.
- `bias_variance(k, grid, n_sets=100, n=56)` – то же для каждой скорости из списка `grid`; для каждой скорости посчитать квадрат смещения $(\bar{\hat y} - f(v))^2$ и разброс (дисперсию прогнозов) и вернуть **средние по сетке** значения: пару `(смещение², разброс)`.

Удобно собрать прогнозы в список строк `preds`: строка – одна выборка, столбец – одна скорость сетки.

In [ ]:
def predictions_at(v0, k, n_sets=100, n=56):
    preds = []
    for s in range(n_sets):
        X, y = make_braking(n=n, seed=1000 + s)             # новая выборка из того же источника
        ...                                     # <-- прогноз для скорости v0
    return preds

def bias_variance(k, grid, n_sets=100, n=56):
    preds = []
    for s in range(n_sets):
        X, y = make_braking(n=n, seed=1000 + s)
        ...                                     # <-- прогнозы для всех скоростей сетки
    bias2, var = 0.0, 0.0
    for j, g in enumerate(grid):
        m, s = ...                              # <-- средний прогноз и его разброс в точке g
        bias2 += ...                            # <-- квадрат смещения
        var += ...                              # <-- дисперсия прогнозов
    return bias2 / len(grid), var / len(grid)


print(f'истинный путь при 100 км/ч: {true_braking(100):.1f} м   ожидается 84.0')
for k in (1, 10, 40):
    m, s = mean_std(predictions_at(100, k))
    print(f'k = {k:2d}: средний прогноз {m:.1f} м, разброс σ = {s:.1f} м')
print('  ожидается: k = 1: 84.0 и 7.9;  k = 10: 84.7 и 3.6;  k = 40: 61.5 и 5.1')

grid = [[v] for v in range(20, 111, 5)]
bv = {}
for k in (1, 3, 5, 10, 20, 40):
    bv[k] = bias_variance(k, grid)
    print(f'k = {k:2d}:  смещение² {bv[k][0]:6.1f}   разброс {bv[k][1]:5.1f}   сумма {sum(bv[k]):6.1f}')
print('  ожидается: k = 1: 0.4 и 43.8;  k = 10: 0.3 и 8.5;  k = 40: 245.9 и 19.7')
noise = sum((3 + 0.05 * g[0]) ** 2 for g in grid) / len(grid)
print(f'шум данных σ² ≈ {noise:.1f}   ожидается 40.9')

In [ ]:
# Готовая ячейка: «мишень» – сто прогнозов для 100 км/ч и U-образная ошибка
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for row, k in enumerate((1, 10, 40)):
    p = predictions_at(100, k)
    ax[0].scatter(p, [row + random.Random(i).uniform(-0.15, 0.15) for i in range(len(p))], s=12,
                  color=[GREY, BLUE, NAVY][row])
ax[0].axvline(true_braking(100), color=DARK_BLUE, linestyle='--', label='истинный путь 84 м')
ax[0].set_yticks([0, 1, 2]); ax[0].set_yticklabels(['k = 1', 'k = 10', 'k = 40'])
ax[0].set_xlabel('прогноз тормозного пути при 100 км/ч, м'); ax[0].legend(); ax[0].grid(alpha=0.3)
ks_bv = sorted(bv)
ax[1].plot(ks_bv, [bv[k][0] for k in ks_bv], 'o-', color=NAVY, label='смещение²')
ax[1].plot(ks_bv, [bv[k][1] for k in ks_bv], 'o-', color=GREY, label='разброс')
ax[1].plot(ks_bv, [sum(bv[k]) for k in ks_bv], 'o-', color=BLUE, label='сумма')
ax[1].set_yscale('log'); ax[1].set_xlabel('k'); ax[1].set_ylabel('м² (логарифмическая шкала)')
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Вопрос Р3.** а) Опишите три облака точек на левом графике словами «смещение» и «разброс». Какая модель – «кучно, но мимо», а какая – «в среднем в центр, но разбросано»? б) Почему при $k = 40$ прогноз для 100 км/ч занижен на 22 м? в) Почему разброс при $k = 1$ почти равен шуму данных $\sigma^2 \approx 41$? г) Разброс снова растет при $k = 20$ и $k = 40$. Попробуйте объяснить, вспомнив, что в выборке всего 56 замеров.

*Ваш ответ:*

**Задание Р4. Бутстрэп: новые выборки из одной.**

В жизни генератора нет – есть одна выборка `Xb_train, yb_train` из 56 замеров. Бутстрэп имитирует новые выборки.

- `bootstrap_sample(X, y, rnd)` – вытянуть $n$ = `len(X)` случайных номеров **с возвращением** (`rnd.randrange(n)` $n$ раз) и вернуть объекты и ответы с этими номерами.
- Проверьте, какая доля различных объектов попадает в бутстрэп-выборку (п. 6 теории).
- Посчитайте для $k \in \{1, 10, 40\}$ средний по сетке разброс прогнозов на 100 бутстрэп-выборках и сравните с разбросом из Р3, где мы брали по-настоящему новые выборки.

In [ ]:
def bootstrap_sample(X, y, rnd):
    n = len(X)
    idx = ...                                   # <-- n случайных номеров с возвращением
    return ...                                  # <-- объекты и ответы с этими номерами


rnd = random.Random(0)
shares = []
for _ in range(1000):
    Xs, _ = bootstrap_sample(list(range(56)), list(range(56)), rnd)
    shares.append(len(set(Xs)) / 56)                # доля различных номеров
print(f'доля различных объектов: {mean_std(shares)[0]:.3f}   ожидается 0.635 (формула: {1 - (1 - 1 / 56) ** 56:.3f})')

boot_var = {}
for k in (1, 10, 40):
    rnd = random.Random(RANDOM_STATE)
    preds = []
    for _ in range(100):
        Xs, ys = ...                            # <-- бутстрэп-выборка из 56 замеров
        ...                                     # <-- прогнозы для сетки скоростей
    boot_var[k] = sum(mean_std([p[j] for p in preds])[1] ** 2 for j in range(len(grid))) / len(grid)
    print(f'k = {k:2d}: разброс по бутстрэпу {boot_var[k]:5.1f},  по новым выборкам (Р3) {bv[k][1]:5.1f}')
print('  ожидается: k = 1: 27.0 и 43.8;  k = 10: 11.7 и 8.5;  k = 40: 18.3 и 19.7')

In [ ]:
# Готовая ячейка: 20 бутстрэп-прогнозов для k = 1 и k = 10
speeds = [[v] for v in range(10, 121)]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for a, k in [(ax[0], 1), (ax[1], 10)]:
    rnd = random.Random(RANDOM_STATE)
    for _ in range(20):
        Xs, ys = bootstrap_sample(Xb_train, yb_train, rnd)
        a.plot([s[0] for s in speeds], knn_regress(Xs, ys, speeds, k), color=BLUE, alpha=0.25, linewidth=1)
    a.plot([s[0] for s in speeds], [true_braking(s[0]) for s in speeds], color=NAVY, linestyle='--', label='истинная зависимость')
    a.scatter([x[0] for x in Xb_train], yb_train, s=10, color=GREY, label='56 замеров')
    a.set_title(f'k = {k}: 20 моделей на бутстрэп-выборках'); a.set_xlabel('скорость, км/ч'); a.grid(alpha=0.3); a.legend()
ax[0].set_ylabel('тормозной путь, м'); plt.tight_layout(); plt.show()

**Вопрос Р4.** а) Почему в бутстрэп-выборке оказывается около 63 % различных объектов, а не 100 %? б) Бутстрэп недооценил разброс при $k = 1$ (27 против 44). Почему? Подсказка: откуда берутся новые значения тормозного пути в бутстрэп-выборке? в) Зачем нужен бутстрэп, если он оценивает разброс неточно?

*Ваш ответ:*

**Задание Р5. Логистическая регрессия со штрафом L2.**

В готовой ячейке – функции логистической регрессии из Б1.3 (`sigmoid`, `linear`, `predict_proba`, `log_loss`, `gradient`) и стандартизация из Б1.6. Напишите по формулам из п. 7 теории:

- `gradient_l2(X, y, w, b, lam)` – градиент log-loss (готовая `gradient`) плюс $\lambda w_j$ к каждой производной по весу; производная по сдвигу $b$ не меняется;
- `loss_l2(y, P, w, lam)` – log-loss плюс $\frac{\lambda}{2}\sum w_j^2$;
- `fit_l2(X, y, lam, lr=2.0, epochs=1000)` – градиентный спуск, как `fit` из Б1.3, но с `gradient_l2`.

Темп обучения 2,0 большой, но для стандартизованных признаков и log-loss он работает быстро и устойчиво; мы его не подбираем.

*Проверка на бумаге.* Если все признаки нулевые, то $P = 0{,}5$, градиент log-loss по весам равен нулю, и остается только $\lambda w$: при $w = (1, -2)$ и $\lambda = 0{,}5$ градиент $(0{,}5; -1)$. Шаг с $\eta = 0{,}1$ умножит веса на $1 - 0{,}1 \cdot 0{,}5 = 0{,}95$: получим $(0{,}95; -1{,}9)$.

In [ ]:
# Готовая ячейка: логистическая регрессия из Б1.3 и стандартизация из Б1.6
def linear(x, w, b):
    return b + sum(wj * xj for wj, xj in zip(w, x))

def sigmoid(t):
    if t >= 0:
        return 1 / (1 + math.exp(-t))
    e = math.exp(t)
    return e / (1 + e)

def predict_proba(X, w, b):
    return [sigmoid(linear(x, w, b)) for x in X]

def log_loss(y, P):
    total = 0
    for yi, p in zip(y, P):
        p = min(max(p, 1e-15), 1 - 1e-15)
        total += yi * math.log(p) + (1 - yi) * math.log(1 - p)
    return -total / len(y)

def gradient(X, y, w, b):
    m = len(X)
    err = [p - yi for p, yi in zip(predict_proba(X, w, b), y)]
    dw = [sum(err[i] * X[i][j] for i in range(m)) / m for j in range(len(w))]
    return dw, sum(err) / m

def standardize(A, B):
    """μ и σ каждого столбца – только по A, применяем к A и B (Б1.6)."""
    A, B = [list(r) for r in A], [list(r) for r in B]
    for j in range(len(A[0])):
        mu, sigma = mean_std([r[j] for r in A])
        sigma = sigma or 1.0
        for r in A: r[j] = (r[j] - mu) / sigma
        for r in B: r[j] = (r[j] - mu) / sigma
    return A, B

def to_labels(P):
    return [1 if p >= 0.5 else 0 for p in P]

print('логистическая регрессия готова')

In [ ]:
def gradient_l2(X, y, w, b, lam):
    dw, db = gradient(X, y, w, b)                        # градиент log-loss из Б1.3
    dw = ...                                    # <-- добавить λ·w_j к каждой производной по весу
    return dw, db

def loss_l2(y, P, w, lam):
    return ...                                  # <-- log-loss плюс штраф

def fit_l2(X, y, lam, lr=2.0, epochs=1000):
    w = [0.0] * len(X[0])
    b = 0.0
    for _ in range(epochs):
        dw, db = ...                            # <-- градиент со штрафом
        w = ...                                 # <-- шаг по весам
        b = ...                                 # <-- шаг по сдвигу
    return w, b


print(gradient_l2([[0, 0], [0, 0]], [1, 0], [1, -2], 0, lam=0.5), '  ожидается ([0.5, -1.0], 0.0)')
dw, db = gradient_l2([[0, 0], [0, 0]], [1, 0], [1, -2], 0, lam=0.5)
print([wj - 0.1 * d for wj, d in zip([1, -2], dw)], '  ожидается [0.95, -1.9]: веса «усохли» в 0.95 раза')
print(round(loss_l2([1, 0], [0.5, 0.5], [1, -2], 0.5), 4), '  ожидается 1.9431 = ln 2 + 0.25 · 5')
w0, b0 = fit_l2([[-2], [-1], [1], [2]], [0, 0, 1, 1], lam=0.0, lr=0.5, epochs=200)
w1, b1 = fit_l2([[-2], [-1], [1], [2]], [0, 0, 1, 1], lam=0.5, lr=0.5, epochs=200)
print(f'вес без штрафа {w0[0]:.2f}, со штрафом λ = 0.5: {w1[0]:.2f}   ожидается 4.01 и 0.71')

**Вопрос Р5.** а) Классы в игрушечном примере разделяются идеально. Что будет с весом без штрафа, если учить не 200, а 10 000 эпох? Почему? б) Почему штрафуют только веса, но не сдвиг $b$? в) Что произойдет при очень большом $\lambda$, например $\lambda = 100$?

*Ваш ответ:*

**Задание Р6. Лишняя емкость и подбор силы штрафа.**

Задача: по скорости и длине отличить пешехода (0) от велосипедиста (1). Беспилотник только начал собирать данные, размечено всего **60 объектов** (`X_pc, y_pc`); еще 76 объектов отложены для итоговой проверки (`X_hold, y_hold`).

Прямая граница у логистической регрессии – это малая емкость. Добавим **степени признаков**: для объекта $(a, c)$ и степени $D$ – все произведения $a^{d-i} c^{i}$ при $1 \le d \le D$, $0 \le i \le d$. Для $D = 2$: $(a, c, a^2, ac, c^2)$, для $D = 5$ – 20 признаков. Модель остается логистической регрессией, но граница становится кривой (п. 2 теории).

1. `poly_features(x, degree)` – список всех степеней для объекта `x = [a, c]` в указанном порядке.
2. `cv_logreg(X, y, folds, degree, lam)` – кросс-валидация логистической регрессии: в каждом раунде построить степени признаков, стандартизировать (`standardize`, параметры только по обучающей части), обучить `fit_l2`, посчитать log-loss на обучающей части и на блоке, точность на блоке и длину вектора весов $\sqrt{\sum w_j^2}$. Вернуть четыре средних по раундам.
3. Кривая по степени при $\lambda = 0$, затем кривая по $\lambda$ при степени 5. Выбираем $\lambda$ с **наименьшим log-loss на валидации**: log-loss (Б1.5) чувствует самоуверенность модели лучше, чем точность.

In [ ]:
# Готовая ячейка: пешеходы и велосипедисты, признаки «скорость» и «длина» после нормализации
pc_idx = [i for i, label in enumerate(y_train) if label in (0, 1)]
small = random.Random(6).sample(pc_idx, 60)
X_pc, y_pc = [Xr_train[i][:2] for i in small], [y_train[i] for i in small]
X_hold = [Xr_train[i][:2] for i in pc_idx if i not in small]
y_hold = [y_train[i] for i in pc_idx if i not in small]
folds_pc = stratified_k_fold_indices(y_pc, 5)
print('размечено:', len(y_pc), ' велосипедистов:', sum(y_pc), ' отложено:', len(y_hold), '  ожидается 60, 32, 76')

In [ ]:
def poly_features(x, degree):
    a, c = x[0], x[1]
    features = []
    for d in range(1, degree + 1):
        for i in range(d + 1):
            ...                                 # <-- произведение a^(d-i) · c^i
    return features

def cv_logreg(X, y, folds, degree, lam):
    tr_loss, val_loss, val_acc, norms = [], [], [], []
    for i in range(len(folds)):
        X_tr, y_tr, X_val, y_val = split_by_fold(X, y, folds, i)
        A = ...                                 # <-- степени признаков обучающей части
        B = ...                                 # <-- степени признаков валидационного блока
        A, B = standardize(A, B)
        w, b = ...                              # <-- обучение со штрафом
        P_tr, P_val = predict_proba(A, w, b), predict_proba(B, w, b)
        ...                                     # <-- log-loss на обучающей части
        ...                                     # <-- log-loss на блоке
        ...                                     # <-- точность на блоке
        norms.append(math.sqrt(sum(wj ** 2 for wj in w)))
    return mean_std(tr_loss)[0], mean_std(val_loss)[0], mean_std(val_acc)[0], mean_std(norms)[0]


print(poly_features([2, 3], 2), '  ожидается [2, 3, 4, 6, 9]')
print(len(poly_features([2, 3], 5)), '  ожидается 20')

print('степень  признаков  log-loss: обучение  валидация   точность')
deg_curve = {}
for D in range(1, 7):
    deg_curve[D] = cv_logreg(X_pc, y_pc, folds_pc, D, lam=0.0)
    tl, vl, va, nw = deg_curve[D]
    print(f'   {D}        {len(poly_features([1, 1], D)):2d}          {tl:.3f}       {vl:.3f}      {va:.3f}')
print('  ожидается: степень 1: 0.197 и 0.239;  степень 5: 0.180 и 0.252;  точность везде 0.933')

In [ ]:
lams = [0.0, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
lam_curve = {}
print('   λ      log-loss: обучение  валидация   точность   длина весов')
for lam in lams:
    lam_curve[lam] = ...                        # <-- кросс-валидация при степени 5
    tl, vl, va, nw = lam_curve[lam]
    print(f'{lam:6.3f}        {tl:.3f}       {vl:.3f}      {va:.3f}      {nw:.2f}')
best_lam = ...                                  # <-- λ с наименьшим log-loss на валидации
print('  ожидается: λ = 0: 0.252;  λ = 0.01: 0.219;  λ = 0.3: 0.495 и точность 0.633')
print('лучшее λ:', best_lam, '  ожидается 0.01')

In [ ]:
# Итоговая проверка на 76 отложенных объектах: λ = 0 против выбранного λ (ячейка готова)
A = [poly_features(x, 5) for x in X_pc]
B = [poly_features(x, 5) for x in X_hold]
A, B = standardize(A, B)
models = {}
for lam in (0.0, best_lam):
    w, b = fit_l2(A, y_pc, lam)
    models[lam] = (w, b)
    P = predict_proba(B, w, b)
    print(f'λ = {lam}:  log-loss {log_loss(y_hold, P):.3f},  точность {accuracy(y_hold, to_labels(P)):.3f}')
print('  ожидается: λ = 0: 0.354 и 0.895;  λ = 0.01: 0.264 и 0.908')

# Сверка с sklearn: наш штраф λ соответствует C = 1 / (λ · n)
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(C=1 / (best_lam * len(A)), max_iter=10000, tol=1e-10).fit(A, y_pc)
w, b = models[best_lam]
print(f'наибольшее расхождение весов со sklearn: {max(abs(a - c) for a, c in zip(w, sk.coef_[0])):.4f}   ожидается 0.0002')

In [ ]:
# Готовая ячейка: граница решения без штрафа и со штрафом
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('p', ['#FFFFFF', '#EAF2FF', '#9CC8FF', '#4C9BFF'])   # вероятность «велосипедист»
g1 = [0.7 * i / 70 for i in range(71)]           # скорость
g2 = [0.45 * i / 70 for i in range(71)]          # длина
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, lam in [(ax[0], 0.0), (ax[1], best_lam)]:
    w, b = models[lam]
    pts = [[x1, x2] for x2 in g2 for x1 in g1]
    _, G = standardize([poly_features(x, 5) for x in X_pc], [poly_features(p, 5) for p in pts])
    Z = [[sigmoid(linear(G[r * 71 + c], w, b)) for c in range(71)] for r in range(71)]
    a.contourf(g1, g2, Z, levels=[0.1 * i for i in range(11)], cmap=cmap)
    a.contour(g1, g2, Z, levels=[0.5], colors=[NAVY], linewidths=2)
    for label, color, name in [(0, GREY, 'пешеход'), (1, NAVY, 'велосипедист')]:
        a.scatter([x[0] for x, t in zip(X_pc, y_pc) if t == label], [x[1] for x, t in zip(X_pc, y_pc) if t == label],
                  s=22, color=color, edgecolor='white', label=name)
    a.set_title(f'λ = {lam}'); a.set_xlabel('скорость (нормализованная)'); a.legend(loc='upper right')
ax[0].set_ylabel('длина (нормализованная)')
fig.suptitle('Степень 5: цвет – вероятность «велосипедист», линия – граница 0,5')
plt.tight_layout(); plt.show()

**Вопрос Р6.** а) Как меняются с ростом степени log-loss на обучающей части и на валидации? Что это значит в терминах емкости? б) Почему при большом $\lambda$ растут обе ошибки, а при $\lambda = 0$ – только валидационная? в) Опишите, чем отличаются две границы на рисунке. Какая из них больше похожа на правило, которое будет работать на новых объектах? г) Почему точность при разных $\lambda$ меняется слабее, чем log-loss?

*Ваш ответ:*

**Задание Р7. Ранняя остановка.**

Вместо штрафа можно просто вовремя остановиться (п. 8 теории). Напишите `fit_early_stopping(X_tr, y_tr, X_val, y_val, lr=0.5, max_epochs=3000, patience=200)`. Темп обучения здесь меньше, чем в Р5 и Р6, чтобы кривая ошибки менялась плавно и минимум было хорошо видно:

1. начать с нулевых весов и делать обычные шаги градиентного спуска **без штрафа** (`gradient` из Б1.3);
2. после каждой эпохи посчитать log-loss на валидационной выборке и добавить в список `history`;
3. если ошибка меньше лучшей – запомнить веса, сдвиг, ошибку и номер эпохи;
4. если лучшая ошибка не улучшалась `patience` эпох подряд – прекратить обучение.

Функция возвращает `best_w, best_b, best_epoch, history`.

Обучаем на 60 объектах со степенью 5, а валидационной выборкой для остановки служат 76 отложенных объектов. Учтите: после этого они уже участвовали в выборе модели, и для честной итоговой оценки понадобилась бы еще одна, тестовая выборка.

In [ ]:
def fit_early_stopping(X_tr, y_tr, X_val, y_val, lr=0.5, max_epochs=3000, patience=200):
    w, b = [0.0] * len(X_tr[0]), 0.0
    best_w, best_b, best_loss, best_epoch = w, b, float('inf'), 0
    history = []
    for epoch in range(1, max_epochs + 1):
        dw, db = gradient(X_tr, y_tr, w, b)
        w = [wj - lr * d for wj, d in zip(w, dw)]
        b = b - lr * db
        val = ...                               # <-- ошибка на валидации после шага
        history.append(val)
        if val < best_loss:
            best_w, best_b, best_loss, best_epoch = ...  # <-- запомнить лучшее
        elif epoch - best_epoch >= patience:
            ...                                 # <-- улучшений нет patience эпох – стоп
    return best_w, best_b, best_epoch, history


A = [poly_features(x, 5) for x in X_pc]
B = [poly_features(x, 5) for x in X_hold]
A, B = standardize(A, B)
w_es, b_es, best_epoch, history = fit_early_stopping(A, y_pc, B, y_hold)
print(f'лучшая эпоха: {best_epoch}, log-loss на валидации {min(history):.3f}, остановились на эпохе {len(history)}')
print('  ожидается: лучшая эпоха 25, log-loss 0.238, остановились на эпохе 225')

In [ ]:
# Готовая ячейка: кривые ошибки за 3000 эпох без остановки
w, b = [0.0] * len(A[0]), 0.0
tr_hist, val_hist = [], []
for epoch in range(3000):
    dw, db = gradient(A, y_pc, w, b)
    w = [wj - 0.5 * d for wj, d in zip(w, dw)]; b -= 0.5 * db
    tr_hist.append(log_loss(y_pc, predict_proba(A, w, b))); val_hist.append(log_loss(y_hold, predict_proba(B, w, b)))
print(f'эпоха 3000: обучение {tr_hist[-1]:.3f}, валидация {val_hist[-1]:.3f}   ожидается 0.189 и 0.339')
plt.figure(figsize=(8, 3.6))
plt.plot(range(1, 3001), tr_hist, color=GREY, label='обучающая выборка (60)')
plt.plot(range(1, 3001), val_hist, color=BLUE, label='валидационная выборка (76)')
plt.axvline(best_epoch, color=NAVY, linestyle='--', label=f'лучшая эпоха {best_epoch}')
plt.xscale('log'); plt.ylim(0.1, 0.4); plt.xlabel('эпоха (логарифмическая шкала)'); plt.ylabel('log-loss')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Вопрос Р7.** а) Почему ошибка на обучающей выборке падает все 3000 эпох, а на валидационной после минимума растет? б) Зачем нужен параметр `patience`, почему нельзя останавливаться при первом же росте ошибки? в) Сравните раннюю остановку и штраф L2: что у них общего? Подсказка: как меняется длина вектора весов с числом эпох?

*Ваш ответ:*

---
### Задания повышенной сложности

1. **Бэггинг деревьев** (Гудфеллоу, 7.11). Напишите модель `bagging_model(X_train, y_train, X_val, params)` по договору из Б2.1: обучить `params['n_models']` деревьев глубины `params['max_depth']`, каждое на своей бутстрэп-выборке, и объявить ответом голосование деревьев (`majority_vote`). Сравните кросс-валидацией на Wine одно дерево глубины 8 и бэггинг из 5, 15 и 25 таких деревьев. Почему выигрыш от бэггинга оказывается скромным? Подсказка: формула 7.51 у Гудфеллоу – что будет, если ошибки моделей сильно похожи?
2. **Оценка без валидации.** Каждое дерево бэггинга не видело около 37 % объектов (п. 6 теории). Для каждого объекта обучающей выборки проголосуйте только деревьями, которые его не видели, и посчитайте точность. Сравните эту оценку с кросс-валидацией.
3. **Случайные признаки.** Сделайте деревья менее похожими: в `best_split` перебирайте не все признаки, а $\lceil\sqrt{n}\rceil$ случайных (для Wine – 4 из 13). Так устроен случайный лес. Помогает ли это бэггингу на Wine?
4. **Штраф L1.** Замените $\frac{\lambda}{2}\sum w_j^2$ на $\lambda\sum |w_j|$ (к градиенту добавляется $\lambda \cdot \mathrm{sign}(w_j)$). Подберите $\lambda$ кросс-валидацией на степенях признаков из Р6 и посчитайте, сколько весов стали почти нулевыми ($|w_j| < 0{,}01$). Почему L1 «выключает» признаки, а L2 только уменьшает веса?
5. **Кривая обучения для логистической регрессии.** Постройте ее для степени 5 при $\lambda = 0$ и при выбранном $\lambda$, беря обучающие выборки из всех 136 пешеходов и велосипедистов. Как штраф меняет разрыв между кривыми?

In [ ]:
# Место для заданий повышенной сложности